In [7]:

from pyspark.sql.functions import *
from pyspark.sql.window import *
from pyspark.sql.types import *
from datetime import date

StatementMeta(, 7e6923b6-0c9e-452c-873d-517ac4109d6e, 13, Finished, Available, Finished, False)

In [3]:
data = [
    ("2010-01-02", 500),
    ("2010-02-03", 1000),
    ("2010-03-04", 1000),
    ("2010-04-05", 1000),
    ("2010-05-06", 1500),
    ("2010-06-07", 1000),
    ("2010-07-08", 1000),
    ("2010-08-09", 1000),
    ("2011-10-10", 1000),
    ("2011-01-02", 500),
    ("2011-02-03", 1000),
    ("2011-03-04", 1000),
    ("2011-04-05", 1000),
    ("2011-05-06", 1550),
    ("2011-06-07", 1100),
    ("2011-07-08", 1100),
    ("2011-08-09", 1000),
]

schema1 = "Sales_Date STRING, Sales_Amount Int"

df = spark.createDataFrame(data,schema1)

df.show()

StatementMeta(, 5ae166f9-6fc2-4b24-b009-8d061741a550, 12, Finished, Available, Finished, False)

+----------+------------+
|Sales_Date|Sales_Amount|
+----------+------------+
|2010-01-02|         500|
|2010-02-03|        1000|
|2010-03-04|        1000|
|2010-04-05|        1000|
|2010-05-06|        1500|
|2010-06-07|        1000|
|2010-07-08|        1000|
|2010-08-09|        1000|
|2011-10-10|        1000|
|2011-01-02|         500|
|2011-02-03|        1000|
|2011-03-04|        1000|
|2011-04-05|        1000|
|2011-05-06|        1550|
|2011-06-07|        1100|
|2011-07-08|        1100|
|2011-08-09|        1000|
+----------+------------+



In [9]:
df1 = df.withColumn("Date",date_format(col("Sales_Date"),'dd-MM-yyyy')).\
   withColumn("Quarter",quarter(col("Sales_Date"))).\
   withColumn("Month",month(col("Sales_Date"))).\
   withColumn("Year",year(col("Sales_Date"))).\
   select("Year","Quarter","Sales_Amount").groupBy("Year","Quarter").\
   agg(sum("Sales_Amount").alias("Total_Sales")).orderBy("Year","Quarter").show()


# Pivoting
# Like to show only Quarter 1 and Quarter 2 sales for every year side by side

pivoted_sales = (
    df1.groupBy("Year")
    .pivot("Quarter", [1, 2])
    .agg(sum("Total_sales"))
    .withColumnRenamed("1", "Q1_sales")
    .withColumnRenamed("2", "Q2_sales")
)
pivoted_sales.show()

StatementMeta(, 5ae166f9-6fc2-4b24-b009-8d061741a550, 62, Finished, Available, Finished, False)

+----+-------+-----------+
|Year|Quarter|Total_Sales|
+----+-------+-----------+
|2010|      1|       2500|
|2010|      2|       3500|
|2010|      3|       2000|
|2011|      1|       2500|
|2011|      2|       3650|
|2011|      3|       2100|
|2011|      4|       1000|
+----+-------+-----------+



AttributeError: 'NoneType' object has no attribute 'groupBy'

In [4]:
schema = StructType([
    StructField("job_status", StringType(), nullable=False),
    StructField("run_date", DateType(), nullable=False)
])

data = [
    ("success", date(2025, 8, 1)),
    ("success", date(2025, 8, 2)),
    ("fail", date(2025, 8, 3)),
    ("fail", date(2025, 8, 4)),
    ("success", date(2025, 8, 13))
]

df_job_runs = spark.createDataFrame(data, schema=schema)
df_job_runs.show()

StatementMeta(, 7e6923b6-0c9e-452c-873d-517ac4109d6e, 7, Finished, Available, Finished, False)

+----------+----------+
|job_status|  run_date|
+----------+----------+
|   success|2025-08-01|
|   success|2025-08-02|
|      fail|2025-08-03|
|      fail|2025-08-04|
|   success|2025-08-13|
+----------+----------+



In [5]:
windowSpec = Window.partitionBy(col("job_status")).orderBy(col("job_status"),col("run_date"))

df_job_runs.withColumn("start_date",col("run_date")).withColumn("end_date",lead(col("run_date")).over(windowSpec)).\
            filter(col("end_date").isNotNull()).show()

StatementMeta(, 7e6923b6-0c9e-452c-873d-517ac4109d6e, 9, Finished, Available, Finished, False)

+----------+----------+----------+----------+
|job_status|  run_date|start_date|  end_date|
+----------+----------+----------+----------+
|      fail|2025-08-03|2025-08-03|2025-08-04|
|   success|2025-08-01|2025-08-01|2025-08-02|
|   success|2025-08-02|2025-08-02|2025-08-13|
+----------+----------+----------+----------+



In [9]:
# Find out Customers who only purchased and Apple and Banana.

data = [
    (111,"Apple"),
    (111,"Banana"),
    (112,"Apple"),
    (113,"Orange"),
    (113,"Apple"),
    (114,"Banana"), 
    (114,"Apple"),
    (114,"Orange"),      

]

schema1 = "userid Int, fruit_name String"

df_fruit = spark.createDataFrame(data,schema1)
df_fruit.show()

StatementMeta(, 7e6923b6-0c9e-452c-873d-517ac4109d6e, 17, Finished, Available, Finished, False)

+------+----------+
|userid|fruit_name|
+------+----------+
|   111|     Apple|
|   111|    Banana|
|   112|     Apple|
|   113|    Orange|
|   113|     Apple|
|   114|    Banana|
|   114|     Apple|
|   114|    Orange|
+------+----------+



In [27]:
df_fruit.groupBy("userid").agg(collect_list(col("fruit_name")).alias("products")).orderBy(col("userid"),\
             col("products")).filter(col("products")==array(lit("Apple"),lit("Banana"))).show()



StatementMeta(, 7e6923b6-0c9e-452c-873d-517ac4109d6e, 36, Finished, Available, Finished, False)

+------+---------------+
|userid|       products|
+------+---------------+
|   111|[Apple, Banana]|
+------+---------------+

